# NB02: Exploratory Data Analysis

## Welfare Scheme Participation Analysis

Explore distributions, correlations, and geographic patterns in:
- **MGNREGA** participation gaps (work demanded vs. work completed)
- **Health insurance** coverage gaps (NFHS-5 data)
- **Census demographics** as determinants

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import MASTER_DATA
from src.feature_engineer import build_features
from src.visualization import (
    setup_plot_style, save_figure, plot_distribution,
    plot_scatter, plot_correlation_heatmap, plot_mgnrega_trends
)

setup_plot_style()
master = pd.read_csv(MASTER_DATA)
df = build_features(master)
print(f'Dataset: {df.shape[0]} districts, {df.shape[1]} features')

## 1. MGNREGA Participation Gap Distribution

In [ ]:
fig = plot_distribution(
    df[df['mgnrega_gap_pct'] >= 0], 'mgnrega_gap_pct',
    'MGNREGA Participation Gap Distribution',
    'Gap (%) = (Demanded - Worked) / Demanded × 100',
    color='#d62728', save_name='mgnrega_gap_distribution'
)
print(f'Median MGNREGA gap: {df["mgnrega_gap_pct"].median():.1f}%')
print(f'Mean MGNREGA gap: {df["mgnrega_gap_pct"].mean():.1f}%')
print(f'Districts with gap > 25%: {(df["mgnrega_gap_pct"] > 25).sum()}')

## 2. Health Insurance Coverage Gap Distribution

In [ ]:
fig = plot_distribution(
    df, 'health_insurance_gap_pct',
    'Health Insurance Coverage Gap Distribution',
    'Gap (%) = 100 - Coverage Rate',
    color='#9467bd', save_name='health_ins_gap_distribution'
)
print(f'Median health insurance gap: {df["health_insurance_gap_pct"].median():.1f}%')
print(f'Mean health insurance gap: {df["health_insurance_gap_pct"].mean():.1f}%')
print(f'Districts with gap > 75%: {(df["health_insurance_gap_pct"] > 75).sum()}')

## 3. Key Demographic Distributions

In [ ]:
demo_features = ['literacy_rate', 'rural_pct', 'sc_pct', 'st_pct',
                 'below_poverty_pct', 'women_literacy_pct', 'vulnerable_pct']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, feat in zip(axes.flat, demo_features):
    if feat in df.columns:
        data = df[feat].dropna()
        ax.hist(data, bins=40, alpha=0.85, edgecolor='white')
        ax.axvline(data.median(), color='red', linestyle='--', linewidth=1.5)
        ax.set_title(feat.replace('_', ' ').title(), fontsize=11)
        ax.set_ylabel('Districts')
for ax in axes.flat[len(demo_features):]:
    ax.set_visible(False)
plt.tight_layout()
save_figure(fig, 'demographic_distributions')
plt.show()

## 4. Correlation Analysis

In [ ]:
corr_features = [
    'mgnrega_gap_pct', 'health_insurance_gap_pct',
    'literacy_rate', 'rural_pct', 'sc_pct', 'st_pct',
    'below_poverty_pct', 'women_literacy_pct',
    'institutional_births_pct', 'electricity_pct',
    'combined_gap_score'
]
existing_corr = [c for c in corr_features if c in df.columns]
fig = plot_correlation_heatmap(
    df, existing_corr,
    'Correlation Matrix: Scheme Gaps & Demographics',
    save_name='correlation_heatmap'
)

## 5. Scatter Plots: Demographics vs. Gaps

In [ ]:
fig = plot_scatter(
    df, 'literacy_rate', 'mgnrega_gap_pct',
    'Literacy Rate vs. MGNREGA Gap',
    'Literacy Rate (%)', 'MGNREGA Gap (%)',
    save_name='literacy_vs_mgnrega_gap'
)

In [ ]:
fig = plot_scatter(
    df, 'rural_pct', 'health_insurance_gap_pct',
    'Rural Population vs. Health Insurance Gap',
    'Rural Population (%)', 'Health Insurance Gap (%)',
    save_name='rural_vs_health_ins_gap'
)

In [ ]:
fig = plot_scatter(
    df, 'vulnerable_pct', 'combined_gap_score',
    'Vulnerable Population vs. Combined Gap Score',
    'Vulnerable (SC+ST) %', 'Combined Gap Score',
    save_name='vulnerable_vs_combined_gap'
)

## 6. MGNREGA National Trends (2011-2021)

In [ ]:
from src.data_loader import load_mgnrega
_, mgnrega_all = load_mgnrega(year='2019-20')

# Rename for trends
mgnrega_all = mgnrega_all.rename(columns={
    'Households that demanded work': 'hh_demanded_work',
    'Households that worked under mahatma gandhi national rural employment guarantee act (mgnrega)': 'hh_worked',
    'Total person days': 'persondays_total',
    'Households that reached a 100 day limit': 'hh_100day',
})

fig = plot_mgnrega_trends(
    mgnrega_all, 'hh_demanded_work',
    'Avg Households Demanding Work',
    save_name='mgnrega_trend_demand'
)

In [ ]:
fig = plot_mgnrega_trends(
    mgnrega_all, 'hh_worked',
    'Avg Households Working Under MGNREGA',
    save_name='mgnrega_trend_worked'
)

## 7. State-Level Summary

In [ ]:
state_summary = df.groupby('state').agg(
    n_districts=('district', 'count'),
    mgnrega_gap=('mgnrega_gap_pct', 'mean'),
    health_ins_gap=('health_insurance_gap_pct', 'mean'),
    combined_gap=('combined_gap_score', 'mean'),
    literacy=('literacy_rate', 'mean'),
    rural=('rural_pct', 'mean'),
    vulnerable=('vulnerable_pct', 'mean'),
).sort_values('combined_gap', ascending=False)

print('Top 10 states by combined gap:')
state_summary.head(10).round(1)

In [ ]:
print('Bottom 10 states by combined gap:')
state_summary.tail(10).round(1)